In [1]:
from ultralytics import YOLO
import os, glob, json, statistics

In [2]:
model = YOLO("yolo26n.pt")          # or yolo26s/m/l/x

In [3]:
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=

In [4]:
m12 =  YOLO("yolo12n.pt")
# check wether we can run it

## Unified RF100 benchmark: baseline -> fine-tune -> val curve -> test

Fine-tuning lives in the standalone script **`finetune_rf100.py`** so it can run outside Jupyter. Per dataset (`cable-damage`, `bone-fracture-7fylg`, `soda-bottles`) it runs the same four steps and writes `../comparison_results/yolo26n_results.json`:

1. **Zero-shot baseline** validation (before fine-tuning)
2. **Fine-tune** for N epochs (default 10; set `--epochs N`)
3. **Per-epoch validation curve** (parsed from the run's `results.csv`)
4. **Final `test`-split** stats

**How to run the fine-tune** (from this directory, this venv):

    .venv/bin/python finetune_rf100.py --epochs 10

The cell below loads the JSON that script produced and shows the baseline vs test table plus the per-dataset validation curves. See `../compare_models.ipynb` for the cross-model comparison.

> **Baseline note:** the pretrained weights emit COCO class ids, while RF100 reuses those ids for different classes, so zero-shot mAP is ~0. It's the honest "before" reference, not a bug.

In [ ]:
# --- Unified benchmark: run fine-tune OUTSIDE the notebook (recommended) ---
# From this directory, with this venv:
#     .venv/bin/python finetune_rf100.py --epochs 10
# Re-run at any depth with --epochs N. To launch from inside Jupyter, uncomment:
# !.venv/bin/python finetune_rf100.py --epochs 10

import json
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS_JSON = Path("../comparison_results/yolo26n_results.json")
data = json.loads(RESULTS_JSON.read_text())
print(f"model={data['model']}  epochs={data['epochs']}  imgsz={data['imgsz']}\n")

# Baseline (zero-shot) vs final test per dataset.
for ds, e in data["datasets"].items():
    b, t = e["baseline_val"], e["final_test"]
    print(f"{ds}")
    print(f"  baseline (zero-shot): mAP50={b['mAP50']:.4f}  mAP50-95={b['mAP50_95']:.4f}")
    print(f"  final test          : mAP50={t['mAP50']:.4f}  mAP50-95={t['mAP50_95']:.4f}")

# Validation curve during fine-tuning, one figure per dataset.
for ds, e in data["datasets"].items():
    curve = e.get("val_curve", [])
    if not curve:
        continue
    ep = [c["epoch"] for c in curve]
    plt.figure(figsize=(8, 5))
    for key in ("mAP50", "mAP50_95"):
        plt.plot(ep, [c.get(key) for c in curve], marker="o", label=key)
    plt.title(f"{data['model']} - {ds}: validation metrics vs epoch")
    plt.xlabel("epoch"); plt.ylabel("value"); plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout(); plt.show()

## One-shot COCO val2017 check (pretrained, no fine-tune)

Sanity reference: evaluate the **pretrained** COCO weights on **COCO `val2017`** (the 5000-image standard split these models were trained on). Unlike the RF100 zero-shot baseline (≈0, because RF100 relabels the class ids), this should reproduce roughly the model's published COCO mAP.

> Uses `val2017`, not `test2017` — COCO's test set has no public labels, so mAP can't be computed on it. Data is prepared once by `../prepare_coco_val.py` (downloads only the val split). The result is stored in `../comparison_results/coco_baseline.json`.

In [1]:
# --- One-shot COCO val2017 sanity check (pretrained weights, NO fine-tune) ---
# All three models are COCO-pretrained, so this should land near their published
# COCO mAP (unlike the ~0 RF100 zero-shot baseline, whose classes don't match COCO).
# Also times batch=1 predict() (warmup + CUDA sync) the same way as the RF-DETR
# benchmark, so FPS is comparable across models.
#
# Prepare the shared COCO val data once (idempotent, ~1 GB; downloads val2017 only):
#     python3 ../prepare_coco_val.py
# To launch from inside Jupyter, uncomment:
# !python3 ../prepare_coco_val.py

import json, time, statistics
from pathlib import Path
import torch
from ultralytics import YOLO

COCO_ROOT = Path.home() / "datasets" / "coco"
COCO_YAML = COCO_ROOT / "coco-val.yaml"
WEIGHTS = "yolo26m.pt"  # pretrained COCO weights

model = YOLO(WEIGHTS)
m = model.val(data=str(COCO_YAML), split="val", verbose=False)
coco = {"mAP50": float(m.box.map50), "mAP50_95": float(m.box.map), "num_images": 5000}

# Batch=1 latency / FPS over all val images (matches the RF-DETR timing loop).
val_images = sorted((COCO_ROOT / "images" / "val2017").glob("*.jpg"))
cuda = torch.cuda.is_available()
for _ in range(3):
    model.predict(str(val_images[0]), verbose=False)
if cuda:
    torch.cuda.synchronize()
lat = []
for p in val_images:
    if cuda:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.predict(str(p), verbose=False)
    if cuda:
        torch.cuda.synchronize()
    lat.append(time.perf_counter() - t0)
mean_s = statistics.mean(lat)
coco.update({"latency_ms_mean": 1000 * mean_s,
             "latency_ms_median": 1000 * statistics.median(lat),
             "fps": 1.0 / mean_s})
print(f"{WEIGHTS} COCO val2017: mAP50={coco['mAP50']:.4f}  mAP50-95={coco['mAP50_95']:.4f}"
      f"  |  {coco['latency_ms_mean']:.1f} ms/img  ({coco['fps']:.1f} FPS)")

BASE = Path("../comparison_results/coco_baseline.json")
allres = json.loads(BASE.read_text()) if BASE.exists() else {}
allres[Path(WEIGHTS).stem] = coco
BASE.write_text(json.dumps(allres, indent=2))
print("saved ->", BASE.resolve())

Ultralytics 8.4.75 🚀 Python-3.11.15 torch-2.12.1+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
YOLO26m summary (fused): 132 layers, 20,411,132 parameters, 0 gradients, 68.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2698.0±550.7 MB/s, size: 212.6 KB)
val: Scanning /home/arina_belova_jetbrains_com/datasets/coco/labels/val2017.cache... 5000 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 806.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 5.1it/s 1:020.1ssss
                   all       5000      36335      0.735      0.627      0.691      0.518
Speed: 0.6ms preprocess, 3.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /home/arina_belova_jetbrains_com/three_od_models/yolov26/runs/detect/val-21
yolo26m.pt COCO val2017: mAP50=0.6907  mAP50-95=0.5180  |  23.7 ms/img  (42.2 FPS)
saved -> /home/arina_belova_jetbrains_com/three_od_models/comparison_results/coco_

## Model size: parameter count

Number of trainable parameters, written to `../comparison_results/model_params.json` so
`compare_models.ipynb` can show it next to the accuracy numbers.

In [2]:
# --- Model size: parameter count ---
import json
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = "yolo26m.pt"
n = sum(p.numel() for p in YOLO(WEIGHTS).model.parameters())
print(f"{WEIGHTS}: {n:,} params ({n/1e6:.2f} M)")

B = Path("../comparison_results/model_params.json")
d = json.loads(B.read_text()) if B.exists() else {}
d[Path(WEIGHTS).stem] = {"num_params": int(n)}
B.write_text(json.dumps(d, indent=2))
print("saved ->", B.resolve())

yolo26m.pt: 21,896,248 params (21.90 M)
saved -> /home/arina_belova_jetbrains_com/three_od_models/comparison_results/model_params.json
